In [ ]:
import numpy as np
import scipy.stats as stats
import pandas as pd
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import random
import json
import time
import pickle
import math

##################################################################
name_to_id = {
    "김포공항": 1,
    "동탄/용인": 2,
    "삼성": 3,
    "서울역": 4,
    "수서": 5,
    "여의도": 6,
    "인천공항": 7,
    "일산": 8,
    "판교": 9,
    "평택": 10,
    "하남/강동": 11
}
##################################################################
json_file_path = 'matrix_t_vp11_6plus.json' # 현재 matrix_t_vp11_6plus.json이 존재 x
with open(json_file_path, 'r') as json_file:
    t = json.load(json_file)

d_hour = []
##################################################################
for time in range(6,24):    
    if time in [7,8] :
        json_file_path = 'demand_morning.json'
        tod = 1
    elif time in [6,9,10,11,12,13,14,15,16,17,20,21,22,23]: 
        json_file_path = 'demand_allday.json'
        tod = 2
    elif time in [18,19]: #18, 19, 20
        json_file_path = 'demand_dinner.json'
        tod = 3
    
    with open(json_file_path, 'r') as json_file:
        demand = json.load(json_file)
        if tod ==1:
            demand = [[val*1.2 for val in row] for row in demand]

######################
    dlevel = 3  ##################################
    d = [[round(value/dlevel/2) for value in row] for row in demand] ####### 2 for half-hour, 10 for reduction
    num_intervals = 30
    capa = 5
    dlist = generateDemand(d,t,num_intervals,capa)      

######################
    if tod == 1:
        df_cleaned = pd.read_csv('AAM수요_오전.csv', encoding='cp949')
    elif tod == 2:
        df_cleaned = pd.read_csv('AAM수요_전일.csv', encoding='cp949')
    else :
        df_cleaned = pd.read_csv('AAM수요_저녁.csv', encoding='cp949')

    df_cleaned['o_name'] = df_cleaned['o_name'].map(name_to_id)
    df_cleaned['d_name'] = df_cleaned['d_name'].map(name_to_id)
    df_cleaned = df_cleaned.dropna(subset=['o_name', 'd_name'])
    df_cleaned = df_cleaned[["승용차","택시","합계","o_name","d_name","time","time_uam"]]

    ######################
    dlist['trvT'] = -1
    dlist['trvT_uam'] = -1
    for row in range(len(dlist)):
        n_value = dlist.loc[row,'n']
        m_value = dlist.loc[row,'m']

        tempdf = df_cleaned[(df_cleaned['o_name'] == n_value) & (df_cleaned['d_name'] == m_value)].copy()
        tempdf.loc[:,'prob']=tempdf['합계']/tempdf['합계'].sum()
        index = np.random.choice(tempdf.index, p=tempdf['prob'])
        chosen_row = tempdf.loc[index]

        dlist.loc[row, 'trvT_uam'] = int(chosen_row['time_uam'])
        dlist.loc[row, 'trvT'] = chosen_row['time']
    d_hour.append(dlist['cnt'].sum())
######################
    filename = f"demand/dlist_{str(dlevel)}_{str(time)}.csv"
    dlist.to_csv(filename, index=False)

In [ ]:
import matplotlib.pyplot as plt
hours = list(range(6, 24))  # Creates [6, 7, 8, ..., 23]
plt.figure(figsize=(10, 6))
plt.plot(hours, d_hour, marker='o', linestyle='-', color='b')
plt.xlabel('Hour of the Day')
plt.ylabel('Value')
plt.title('Data Plot from 6 AM to 11 PM')
plt.xticks(hours)  # Ensure all hours are shown on x-axis

plt.grid(True)
plt.show()
sum(d_hour)

In [9]:
import numpy as np
import scipy.stats as stats
import pandas as pd
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import random
import json
import time
import pickle
import math

###################################
json_file_path = 'matrix_t_vp11_6plus.json'
with open(json_file_path, 'r') as json_file:
    t = json.load(json_file)

json_file_path = 'matrix_dist_vp11_6plus.json'
with open(json_file_path, 'r') as json_file:
    dist = json.load(json_file)

###############
dlevel = 3
num_intervals = 30
# numVeh = 10
# max_battery =  100 #km
# chargeByTime = int(max_battery/num_intervals)
capa = 5
alpha = 100
vpCapacity = 100
################

C:\Users\USER\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\USER\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [10]:
for numVeh in [40,50,60]:#, 20, 30]:
    for max_battery in [100,160,220]:#[100, 160, 220]:
        chargeByTime = int(max_battery/num_intervals)
        for time in range(6,24):
            print(time)
            filename = f"demand/dlist_{str(dlevel)}_{str(time)}.csv"
            dlist = pd.read_csv(filename)    
            chargeTime = []
        #     curTime = [0 for i in range(numVeh)]
        ###############
            if time == 6:
                match, df = generatePickDel(dlist)
                curNode = [0 for i in range(numVeh)] #[6, 6]
                curBattery = [max_battery for i in range(numVeh)] #[6, 6]
                curTime = [0 for i in range(numVeh)]
                lastBattery = curBattery

            else :
                curNode, curBattery, curTime, chargeTime = vehicleUpdate(log,match,lastBattery,numVeh,num_intervals)
                if [time > 60 for time in curTime]==True:
                    asd +=1
                tempNode = curNode

                match, df = generatePickDel(dlist)

                temp_charge = [x * chargeByTime for x in chargeTime]
                curBattery = [min(x + y,max_battery) for x, y in zip(curBattery, temp_charge)]
                curBattery = [int(x) for x in curBattery]

                curNode, match = newMatch(match, tempNode)
                lastBattery = curBattery

            new_t = addzero(newmat(t,match))
            new_dist = addzero(newmat(dist,match))

            dd = transform_demand(df,match,curNode)
            df, tw = transform_tw(df,match, numVeh, curTime, curNode,num_intervals,alpha)
        ###############
            if __name__ == "__main__":
                log, manager, routing, solution, data, dropped_nodes = main()
        ###############    
            save_data_to_json(log, match, dropped_nodes, lastBattery, curBattery, chargeTime, df, new_t, new_dist, dd, tw , dlist, 
                              data, f"result/w15_output_{str(numVeh)}_{str(max_battery)}_{str(dlevel)}_{str(time)}.pickle")


6


100%|█████████████████████████████████████████████████████| 342/342 [00:00<00:00, 3440.71it/s]


Objective: 12221
Dropped nodes:

Route for vehicle 0:
Node:0 Time(0,0) Distance:0 Load:0 
 -> Node:8 Time(0,0) Distance:0 Load:0 
 -> Node:62 Time(14,14) Distance:29 Load:1 
 -> Node:39 Time(14,14) Distance:29 Load:2 
 -> Node:208 Time(28,28) Distance:55 Load:5 
 -> Node:175 Time(42,42) Distance:80 Load:2 
 -> Node:231 Time(42,42) Distance:80 Load:1 
 -> Node:0 Time(42,42) Distance:80 Load:0)
Time of the route: 42min
Distance of the route: 80km
Load of the route: 0


Route for vehicle 1:
Node:0 Time(0,0) Distance:0 Load:0 
 -> Node:44 Time(8,9) Distance:0 Load:0 
 -> Node:53 Time(9,9) Distance:0 Load:1 
 -> Node:73 Time(17,17) Distance:5 Load:3 
 -> Node:78 Time(28,28) Distance:20 Load:4 
 -> Node:222 Time(28,28) Distance:20 Load:5 
 -> Node:93 Time(28,28) Distance:20 Load:3 
 -> Node:213 Time(28,28) Distance:20 Load:4 
 -> Node:242 Time(28,28) Distance:20 Load:3 
 -> Node:262 Time(53,53) Distance:86 Load:2 
 -> Node:247 Time(53,53) Distance:86 Load:1 
 -> Node:0 Time(53,53) Distance:8

100%|█████████████████████████████████████████████████████| 472/472 [00:00<00:00, 3256.03it/s]


Objective: 2368293
Dropped nodes: 2 7 10 14 15 18 21 24 25 26 29 33 36 38 39 40 42 43 44 45 46 48 49 50 54 55 56 60 61 63 65 66 67 69 70 71 73 74 75 82 86 87 88 89 90 92 93 96 97 99 104 106 108 109 110 111 116 117 122 123 124 127 128 135 136 137 138 139 140 141 142 145 146 148 149 150 151 152 153 156 158 159 160 161 162 163 164 169 170 171 173 174 175 178 180 181 182 183 184 185 186 187 189 190 192 193 196 198 199 201 202 203 205 206 207 208 209 212 216 224 226 227 228 231 234 238 239 240 243 247 250 252 253 254 256 257 258 259 260 262 263 264 268 269 270 274 275 277 279 280 283 285 286 287 289 290 291 298 302 303 304 305 306 308 309 312 313 315 320 322 324 325 326 327 332 333 338 339 340 343 344 351 352 353 354 355 356 357 358 361 362 364 365 366 367 368 369 372 374 375 376 377 378 379 380 385 386 387 389 390 391 394 396 397 398 399 400 401 402 404 405 407 408 411 413 414 416 417 418 420 421 422 423 424 427 431

Route for vehicle 0:
Node:433 Time(12,12) Distance:0 Load:0 
 -> Node:0 T

...

Objective: 294129
Dropped nodes: 2 7 35 38 50 80 84 93 95 96 97 123 141 159 194 195 199 211 243 247 256 258 259 260 261 288 306 324

Route for vehicle 0:
Node:331 Time(0,0) Distance:0 Load:0 
 -> Node:43 Time(10,10) Distance:13 Load:0 
 -> Node:48 Time(10,10) Distance:13 Load:1 
 -> Node:209 Time(22,22) Distance:33 Load:2 
 -> Node:204 Time(22,22) Distance:33 Load:1 
 -> Node:85 Time(30,30) Distance:38 Load:0 
 -> Node:94 Time(30,30) Distance:38 Load:3 
 -> Node:257 Time(44,44) Distance:65 Load:5 
 -> Node:248 Time(44,44) Distance:65 Load:3 
 -> Node:0 Time(44,44) Distance:65 Load:0)
Time of the route: 44min
Distance of the route: 65km
Load of the route: 0


Route for vehicle 1:
Node:332 Time(0,0) Distance:0 Load:0 
 -> Node:81 Time(15,17) Distance:29 Load:0 
 -> Node:15 Time(15,17) Distance:29 Load:3 
 -> Node:173 Time(26,28) Distance:46 Load:5 
 -> Node:244 Time(26,28) Distance:46 Load:3 
 -> Node:113 Time(26,28) Distance:46 Load:0 
 -> Node:158 Time(28,28) Distance:46 Load:1 
 -> No

100%|█████████████████████████████████████████████████████| 406/406 [00:00<00:00, 3497.75it/s]


Objective: 475893
Dropped nodes: 13 15 23 26 27 31 39 42 44 48 57 63 70 77 85 87 106 110 120 121 127 136 156 179 183 193 196 197 201 209 212 214 218 227 233 241 249 257 259 278 282 293 294 300 309 329

Route for vehicle 0:
Node:347 Time(14,14) Distance:0 Load:0 
 -> Node:0 Time(14,14) Distance:0 Load:0)
Time of the route: 14min
Distance of the route: 0km
Load of the route: 0


Route for vehicle 1:
Node:348 Time(20,20) Distance:0 Load:0 
 -> Node:118 Time(20,20) Distance:0 Load:0 
 -> Node:1 Time(20,20) Distance:0 Load:1 
 -> Node:114 Time(30,30) Distance:13 Load:2 
 -> Node:287 Time(42,42) Distance:33 Load:3 
 -> Node:286 Time(42,42) Distance:33 Load:2 
 -> Node:291 Time(52,52) Distance:45 Load:1 
 -> Node:0 Time(52,52) Distance:45 Load:0)
Time of the route: 52min
Distance of the route: 45km
Load of the route: 0


Route for vehicle 2:
Node:349 Time(8,8) Distance:0 Load:0 
 -> Node:10 Time(12,12) Distance:0 Load:0 
 -> Node:159 Time(36,36) Distance:64 Load:1 
 -> Node:244 Time(36,36) Di

100%|█████████████████████████████████████████████████████| 436/436 [00:00<00:00, 3204.13it/s]


Objective: 1133994
Dropped nodes: 2 3 10 11 14 15 20 25 28 32 33 35 38 41 44 46 48 49 50 52 54 55 57 60 66 73 76 80 81 89 90 96 97 102 105 108 109 110 111 112 116 119 123 125 129 132 138 139 140 145 146 150 152 163 181 183 190 195 196 198 204 209 210 213 217 218 220 223 226 229 231 233 234 235 237 240 241 243 246 252 259 262 266 267 275 276 282 284 285 290 293 296 297 298 299 300 304 307 311 313 317 320 326 327 328 333 334 338 340 351 369 371

Route for vehicle 0:
Node:377 Time(0,0) Distance:0 Load:0 
 -> Node:5 Time(15,15) Distance:32 Load:0 
 -> Node:98 Time(15,15) Distance:32 Load:1 
 -> Node:83 Time(15,15) Distance:32 Load:2 
 -> Node:77 Time(15,15) Distance:32 Load:3 
 -> Node:75 Time(15,15) Distance:32 Load:4 
 -> Node:286 Time(34,34) Distance:77 Load:5 
 -> Node:269 Time(34,34) Distance:77 Load:4 
 -> Node:261 Time(34,34) Distance:77 Load:3 
 -> Node:193 Time(34,34) Distance:77 Load:2 
 -> Node:263 Time(53,53) Distance:121 Load:1 
 -> Node:0 Time(53,53) Distance:121 Load:0)
Time

In [ ]:
############################

In [ ]:
numVeh = 30
max_battery =  100 #km
chargeByTime = int(max_battery/num_intervals)

In [ ]:
time = 6
filename = f"demand/dlist_{str(dlevel)}_{str(time)}.csv"
dlist = pd.read_csv(filename)   

chargeTime = []
match, df = generatePickDel(dlist)
curNode = [0 for i in range(numVeh)] #[6, 6]
curBattery = [max_battery for i in range(numVeh)] #[6, 6]
curTime = [0 for i in range(numVeh)]
lastBattery = curBattery

new_t = addzero(newmat(t,match))
new_dist = addzero(newmat(dist,match))

dd = transform_demand(df,match,curNode)
df, tw = transform_tw(df,match, numVeh, curTime, curNode,num_intervals,alpha)
log, manager, routing, solution, data, dropped_nodes = main()

In [ ]:
time += 1
filename = f"demand/dlist_{str(dlevel)}_{str(time)}.csv"
dlist = pd.read_csv(filename)    
# chargeTime = []

curNode, curBattery, curTime, chargeTime = vehicleUpdate(log,match,lastBattery,numVeh,num_intervals)
print(curTime)
print(curNode)

In [ ]:
tempNode = curNode
match, df = generatePickDel(dlist)
temp_charge = [x * chargeByTime for x in chargeTime]
curBattery = [min(x + y,max_battery) for x, y in zip(curBattery, temp_charge)]
curBattery = [int(x) for x in curBattery]

curNode, match = newMatch(match, tempNode)
lastBattery = curBattery
print(curBattery)
print(curNode)

In [ ]:
df[df['depT']+df['maxRideT']>60]

In [ ]:
new_t = addzero(newmat(t,match))
new_dist = addzero(newmat(dist,match))

dd = transform_demand(df,match,curNode)
df, tw = transform_tw(df,match, numVeh, curTime, curNode,num_intervals,alpha)

In [ ]:
log, manager, routing, solution, data, dropped_nodes = main()

In [ ]:
##############################################

In [1]:
def create_data_model(new_dist,new_t,df,numVeh,dd,curNode,curTime,curBattery,tw,capa, vpCapacity):
    data = {}
    data['distance_matrix'] = new_dist
    data['time_matrix'] = new_t
    data['pickups_deliveries'] = df[['n','m']].values.tolist()
    data['num_vehicles'] = numVeh #############
    data["demands"] = dd #############
    
    data['starts'] = curNode
    data["battery"] = curBattery
    
    data['ends'] = [0 for i in range(numVeh)] #[6, 6]    
    data["node2visit"] = list(set(item for sublist in data['pickups_deliveries'] for item in sublist))    
    data["vehicle_capacities"] = [capa for i in range(numVeh)] #[6, 6]

    data['maxRideT'] = df[['n', 'maxRideT']].set_index('n')['maxRideT'].apply(lambda x: math.ceil(x)).to_dict() #df[['n', 'maxRideT']].set_index('n').to_dict()['maxRideT']

    data['time_windows'] = tw
    data['vehicle_availability'] = [max(time - num_intervals,0) for time in curTime]
    data['vpCapacity'] = vpCapacity
    return data

In [2]:
def vehicleUpdate(log,match,lastBattery,numVeh,num_intervals):
    new_row = {'initial': 0, 'new': 0}
    match = pd.concat([pd.DataFrame([new_row]), match], ignore_index=True)
    match = match.drop_duplicates(subset=['initial', 'new'])

    match = match.reset_index(drop=True)

    curNode = log['Visited Nodes'].apply(lambda x: x[-2] if len(x) >= 2 else 0)
    curNode = pd.DataFrame({'new':curNode})
    curNode = curNode.merge(match, on='new', how='left')
    curNode['initial'] = curNode['initial'].fillna(0).astype(int)
    curNode = curNode['initial'].tolist()
    
    curTime = log['Visited Time'].apply(extract_last_value).tolist()

    distanceTravelled = log['Distance'].diff()
    distanceTravelled.iloc[0] = log['Distance'].iloc[0]
    curBattery = [a - b for a, b in zip(lastBattery, distanceTravelled.tolist())]

    chargeTime = [0 for _ in range(numVeh)]
    for val in range(len(curTime)):
        if curTime[val] < num_intervals:
            chargeTime[val] = num_intervals - curTime[val]
    return curNode, curBattery, curTime, chargeTime

def extract_last_value(lst):
    return lst[-1][-1] if lst else None

In [3]:
def transform_tw(df, match, numVeh, curTime, tempNode, num_intervals, alpha):
    tw4start = []
    node4start =[]
    if curTime:        
        ind = [index for index, value in enumerate(curTime) if value > num_intervals]
        if ind:
            startNode = tempNode[-numVeh:]
            for index, value in enumerate(curTime):
                if value > num_intervals:
                    node4start.append(startNode[index]) 
                    tw4start.append(curTime[index])    
                    
######################
    def calculate_val(row):
        temp = row['trvT']/2-row['trvT_uam'] 
        if temp >= 15:
            return 15 #10*num_intervals/30 if random.randint(1, 100) <= alpha else 1 ############ 확률?
        else :
            return math.floor(temp) #5*num_intervals/30 if random.randint(1, 100) <= alpha else 0 ############ 확률?

    df['val'] = df.apply(calculate_val, axis=1)
    df['maxRideT'] = df['arrT']-df['depT']+df['val'] # + 5*num_intervals/30 ######## MaxRideT 개념
    df['maxRideT'] = df['maxRideT'].apply(math.ceil) ######## MaxRideT 개념
######################
    tw = [(0, 500)]
    for n in range(1,max(match['new'])+1):
        if n in df['n'].values:
            start_time = int(round(df[df['n'] == n]['depT'].values[0]))
            val = df[df['n'] == n]['val'].values[0]
            end_time = int(start_time + val) # + 5*num_intervals/30)) ######## 5분 추가 맞음?
            tw.append((start_time, end_time))
        elif n in df['m'].values:
            start_time = int(math.ceil(df[df['m'] == n]['arrT'].values[0]))
            val = df[df['m'] == n]['val'].values[0]
            end_time = int(start_time + val) # + 5*num_intervals/30)) ######## 5분 추가 맞음?
            tw.append((start_time, end_time))
        elif n in node4start:
            tw.append((tw4start[node4start.index(n)] - num_intervals, 500))
        else: 
            tw.append((0, 500))              
    return df, tw

In [4]:
def main():
    # Instantiate the data problem.
    data = create_data_model(new_dist,new_t,df,numVeh,dd,curNode,curTime,curBattery,tw,capa, vpCapacity)
    
    # Create the routing index manager.
    manager = pywrapcp.RoutingIndexManager(
        len(data["distance_matrix"]), data["num_vehicles"], data["starts"], data["ends"]#data["depot"]
    )

    # Create Routing Model.
    routing = pywrapcp.RoutingModel(manager)

    #########################################    
    # Define cost of each arc.
    def distance_callback(from_index, to_index):
        # Convert from routing variable Index to distance matrix NodeIndex.
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data["distance_matrix"][from_node][to_node]
    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    #########################################    
    # Add Distance constraint.
    dimension_name = "Distance"
    routing.AddDimensionWithVehicleCapacity(
        transit_callback_index,
        0,  # no slack
        data["battery"],  ########################### vehicle maximum travel distance
        True,  # start cumul to zero
        dimension_name,
    )
    distance_dimension = routing.GetDimensionOrDie(dimension_name)
    distance_dimension.SetGlobalSpanCostCoefficient(100)
    
    #########################################        
    # Add Capacity constraint.
    def demand_callback(from_index):
        # Convert from routing variable Index to demands NodeIndex.
        from_node = manager.IndexToNode(from_index)
        return data["demands"][from_node]
    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,  # null capacity slack
        data["vehicle_capacities"],  ##########################
        True,  # start cumul to zero
        "Capacity",
    )

    #########################################    
    # Add Time Windows constraint. SAME
    def time_callback(from_index, to_index):
        # Convert from routing variable Index to time matrix NodeIndex.
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return data['time_matrix'][from_node][to_node]

    time_callback_index = routing.RegisterTransitCallback(time_callback)
    time = 'Time'
    routing.AddDimension(
        time_callback_index,
        100,  # allow waiting time
        100,  # maximum time per vehicle
        False,  # Don't force start cumul to zero.
        time)
    time_dimension = routing.GetDimensionOrDie(time)   
    
    #########################################        
    # Add time window constraints for each location except depot.
    for location_idx, time_window in enumerate(data["time_windows"]):
        if location_idx == 0 :
            continue
        index = manager.NodeToIndex(location_idx)
        time_dimension.CumulVar(index).SetRange(time_window[0], time_window[1])

    # Add time window constraints for each vehicle start node.
    depot_idx = data["starts"]
    for vehicle_id in range(data["num_vehicles"]):
        index = routing.Start(vehicle_id)
        time_dimension.CumulVar(index).SetRange(
            data["time_windows"][depot_idx[vehicle_id]][0], data["time_windows"][depot_idx[vehicle_id]][1]
        )

    # Instantiate route start and end times to produce feasible times.
    for i in range(data["num_vehicles"]):
        routing.AddVariableMinimizedByFinalizer(
            time_dimension.CumulVar(routing.Start(i))
        )
        routing.AddVariableMinimizedByFinalizer(time_dimension.CumulVar(routing.End(i)))

    #########################################    
    # Allow to drop nodes
    penalty = 0
    for node in range(1, len(data["distance_matrix"])):
        routing.AddDisjunction([manager.NodeToIndex(node)], penalty)

    penalty = 10000
    for node in data["node2visit"]: 
        routing.AddDisjunction([manager.NodeToIndex(node)], penalty)
    
    #########################################    
    # Define Transportation Requests.
    for request in data["pickups_deliveries"]:
        pickup_index = manager.NodeToIndex(request[0])
        delivery_index = manager.NodeToIndex(request[1])
        routing.AddPickupAndDelivery(pickup_index, delivery_index)
        routing.solver().Add(
            routing.VehicleVar(pickup_index) == routing.VehicleVar(delivery_index)
        )
        routing.solver().Add(
            distance_dimension.CumulVar(pickup_index)
            <= distance_dimension.CumulVar(delivery_index)
        )
    
    #########################################
    # Add Ride Time Constraint for DARP
    for request in data['pickups_deliveries']:
        pickup_index = manager.NodeToIndex(request[0])
        delivery_index = manager.NodeToIndex(request[1])
        pickup_time = time_dimension.CumulVar(pickup_index)
        delivery_time = time_dimension.CumulVar(delivery_index)
        max_ride_time = data['maxRideT'][request[0]]  # Assuming request[0] is the pickup node and has the ride time information
        routing.solver().Add(delivery_time <= pickup_time + max_ride_time) ################# maxRideT

    #########################################
    # Add Vehicle Availability Constraint.
    for vehicle_id in range(data['num_vehicles']):
        availability_time = data['vehicle_availability'][vehicle_id]
        index = routing.Start(vehicle_id)
        # Ensure vehicle is available starting from its availability time+1
        routing.solver().Add(time_dimension.CumulVar(index) >= availability_time) #################### loading/unloading 3 min?

            
    #########################################    
    # Setting first solution heuristic.
    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.AUTOMATIC
    )

    # Solve the problem.
    solution = routing.SolveWithParameters(search_parameters)

    # Print solution on console.
    if solution:
        log = print_solution(manager, routing, solution)

    dropped_nodes = []
    for node in range(routing.Size()):
        if routing.IsStart(node) or routing.IsEnd(node):
            continue
        if solution.Value(routing.NextVar(node)) == node:
            dropped_nodes.append(manager.IndexToNode(node))
    
    return log, manager, routing, solution, data, dropped_nodes

In [5]:
def generateDemand(d,t,num_intervals,capa):
    dlist = []

    for n in range(0,len(d)):
        for m in range(0,len(d)):
            if n == m:
                continue

            total_demand = round(d[n][m],0)
            average_rate = total_demand / num_intervals

            demand_counts = []

            for interval in range(num_intervals):
                demand = np.random.poisson(average_rate)
                
                while demand > 0:
                    split_demand = min(demand, capa)
                    dlist.append({'n': n + 1, 'm': m + 1, 'cnt': split_demand, 'depT': interval, 'arrT': interval + t[n][m]})
                    demand -= split_demand

    dlist = pd.DataFrame(dlist)
    dlist = dlist[dlist['cnt']!=0]
    dlist = dlist.sort_values(by='depT').reset_index(drop=True)
    return dlist

def generatePickDel(dlist):
    df = dlist[['n','m']]
    df = pd.DataFrame(df)
    temp = df['n'].to_list() + df['m'].to_list()

    new_num = max(temp)+1
    for i in range(1,len(temp)):
        priorvalues = temp[:i]
        if temp[i] in priorvalues:
            temp[i] = new_num
            new_num = new_num+1
    ini_temp = df['n'].to_list() + df['m'].to_list()
    match = pd.DataFrame({'initial': ini_temp, 'new': temp})
    #match = match[match['initial'] != match['new']].reset_index(drop=True)
    df['n'] = match[:int(len(temp)/2)]['new'].to_list()
    df['m'] = match[int(len(temp)/2):]['new'].to_list()
    df = pd.concat([df,dlist[['cnt','depT','arrT','trvT','trvT_uam']]], axis=1)
    return(match,df)

In [6]:
from tqdm import tqdm

def newmat(t, match):
    t = np.array(t)
    max_new = max(match['new'])
    new_t = np.zeros((max_new, max_new), dtype=float)

    # Create lookup arrays for faster access
    initial_lookup = match.set_index('new')['initial'].to_dict()
    new_indices = match['new'].values

    for i in tqdm(range(max_new)):
        if any(match['initial'][match['new'] == i+1] == 0):
            new_t[i, :] = 0
            new_t[:, i] = 0
            continue
        
        replace_i = initial_lookup.get(i+1, i+1)
        if replace_i == 0:
            new_t[i, :] = 0
            new_t[:, i] = 0
            continue
        
        for j in range(max_new):
            if i == j:
                continue
            replace_j = initial_lookup.get(j+1, j+1)
            if replace_j == 0:
                new_t[i, j] = 0
                new_t[j, i] = 0
            else:
                new_t[i, j] = t[replace_i-1, replace_j-1]
    return new_t

def addzero(t):
    row_of_zeros = np.zeros((1, t.shape[1]), dtype=int)
    t = np.vstack((row_of_zeros, t))
    column_of_zeros = np.zeros((t.shape[0], 1), dtype=int)
    t = np.hstack((column_of_zeros, t))
    return t

def transform_demand(df, match, curNode):
    max_n = max(match['new']) + 1
    
    # Create dictionaries for quick lookup
    n_to_cnt = df.set_index('n')['cnt'].to_dict()
    m_to_cnt = df.set_index('m')['cnt'].to_dict()
    
    # Initialize the demand array
    dd = np.full(max_n, 1000, dtype=int)
    
    # Update the demand array based on the conditions
    n_keys = np.array(list(n_to_cnt.keys()))
    m_keys = np.array(list(m_to_cnt.keys()))
    curNode = np.array(curNode)

    dd[n_keys] = [n_to_cnt[k] for k in n_keys]
    dd[m_keys] = [-m_to_cnt[k] for k in m_keys]
    dd[curNode] = 0

    # Ensure the first element is 0
    dd[0] = 0

    return dd.tolist()

# def transform_demand(df,match,curNode):
#     dd = [0]
#     for n in range(1,max(match['new']+1)):
#         if n in df['n'].values:
#             dd.append(df[df['n']==n]['cnt'].values[0])
#         elif n in df['m'].values:
#             dd.append(-df[df['m']==n]['cnt'].values[0])
#         elif n in curNode: 
#             dd.append(0)              
#         else:
#             dd.append(1000)              
#     return dd  

def newMatch(match,tempNode):
    N = max(match['new'])+1
    new_curNode = tempNode.copy()
    for ind in range(0,len(tempNode)):
        if tempNode[ind] == 0:
            new_curNode[ind] = 0
        else:
            new_curNode[ind] = N
            N += 1
    temp = {'initial': tempNode, 'new': new_curNode}
    temp = pd.DataFrame(temp).drop_duplicates().reset_index(drop=True)
    match = pd.concat([match,temp]).reset_index(drop=True)
    match = match.astype(int)

    return new_curNode, match

In [7]:
def save_data_to_json(log, match, dropped_nodes, lastBattery, curBattery, chargeTime, df, new_t, new_dist, dd, tw, dlist, data, filename):
    data_to_save = {
        "log": log.to_dict(orient='split'),
        "match": match.to_dict(orient='split'),
        "dropped_nodes": dropped_nodes,
        "lastBattery": lastBattery,
        "curBattery": curBattery,
        "chargeTime": chargeTime,
        "df": df,
        "new_t": new_t,
        "new_dist": new_dist,
        "dd" : dd,
        "tw": tw,
        "dlist": dlist,
        "data": data
    }

    with open(filename, 'wb') as pickle_file:
        pickle.dump(data_to_save, pickle_file)

In [8]:
def print_solution(manager, routing, solution):
    print(f'Objective: {solution.ObjectiveValue()}')
    route_data = []

    # Display dropped nodes.
    dropped_nodes = 'Dropped nodes:'
    for node in range(routing.Size()):
        if routing.IsStart(node) or routing.IsEnd(node):
            continue
        if solution.Value(routing.NextVar(node)) == node:
            dropped_nodes += ' {}'.format(manager.IndexToNode(node))
    print(dropped_nodes)
    
    # Print routes
    time_dimension = routing.GetDimensionOrDie('Time')
    distance_dimension = routing.GetDimensionOrDie('Distance')
    capacity_dimension = routing.GetDimensionOrDie('Capacity')
    total_time = 0
    total_distance = 0
    total_load=0
    
    for vehicle_id in range(manager.GetNumberOfVehicles()):
        visited_nodes = []
        visited_time = []    
        visited_load = []
        index = routing.Start(vehicle_id)
        plan_output = f'\nRoute for vehicle {vehicle_id}:\n'
        
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            time_var = time_dimension.CumulVar(index)
            distance_var = distance_dimension.CumulVar(index)
            capacity_var = capacity_dimension.CumulVar(index)
            plan_output += 'Node:{0} Time({1},{2}) Distance:{3} Load:{4} \n -> '.format(
                node_index,
                solution.Min(time_var), solution.Max(time_var),
                solution.Value(distance_var),
                solution.Value(capacity_var))
            visited_nodes.append(node_index)
            visited_time.append((solution.Min(time_var), solution.Max(time_var)))            
            visited_load.append(solution.Value(capacity_var))
            index = solution.Value(routing.NextVar(index))
            
        node_index = manager.IndexToNode(index)
        time_var = time_dimension.CumulVar(index)
        distance_var = distance_dimension.CumulVar(index)
        capacity_var = capacity_dimension.CumulVar(index)
        plan_output += 'Node:{0} Time({1},{2}) Distance:{3} Load:{4})\n'.format(
            manager.IndexToNode(index),
            solution.Min(time_var), solution.Max(time_var),
            solution.Value(distance_var),
            solution.Value(capacity_var))
        plan_output += 'Time of the route: {}min\n'.format(solution.Min(time_var))
        plan_output += 'Distance of the route: {}km\n'.format(solution.Value(distance_var))
        plan_output += 'Load of the route: {}\n'.format(solution.Value(capacity_var))
        print(plan_output)

        total_time += solution.Min(time_var)
        total_distance += solution.Value(distance_var)
        total_load += solution.Value(capacity_var)
        
        visited_nodes.append(node_index)
        visited_time.append((solution.Min(time_var), solution.Max(time_var)))
        visited_load.append(solution.Value(capacity_var))
        
        route_info = {
            'Vehicle': vehicle_id,
            'Visited Nodes': visited_nodes,
            'Visited Time': visited_time,
            'Distance': total_distance,
            'Load': visited_load
        }

        route_data.append(route_info)       
    df = pd.DataFrame(route_data)
#     print('Total time of all routes: {}min'.format(total_time))
#     print('Total distance of all routes: {}km'.format(total_distance))
#     print('Total load of all routes: {} \n'.format(total_load))
#     print(df)
    
    return df